# Transformer Text Generation — Mini Project

This notebook converts the previous Transformer translation structure into a **decoder-only Transformer text generation model**.

### Main concept

**Previous tokens → Transformer → next-token prediction**

Example:

`Artificial intelligence` → `helps`

Then:

`Artificial intelligence helps` → `analyze`

The model generates text one token at a time.


## 1. Imports

In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import tensorflow as tf
import pandas as pd

from tensorflow.keras.layers import (
    Embedding,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


## 2. Load the Text Generation Dataset

In [4]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

DATASET_PATH = "transformer_text_generation_500_rows.csv"

dataset = pd.read_csv(DATASET_PATH)

texts = dataset["text"].astype(str).tolist()

print("Training texts:", len(texts))

for i in range(min(5, len(texts))):
    print(f"{i + 1}. {texts[i]}")
    print()


Training texts: 500
1. Natural language processing helps build intelligent applications for document processing. Embeddings represent tokens as numerical vectors in a continuous space.

2. Computer vision allows answer questions from a knowledge base for developers and researchers. The learning rate controls how strongly model parameters are updated during training.

3. Python programming helps analyze large amounts of data for developers and researchers. Embeddings represent tokens as numerical vectors in a continuous space.

4. Large language models can improve detect patterns that are difficult to find manually in modern software applications. Validation data helps measure how well the model generalizes to unseen examples.

5. Neural networks makes it possible to answer questions from a knowledge base for automation workflows. Embeddings represent tokens as numerical vectors in a continuous space.



## 3. Tokenizer

For text generation, we use **one tokenizer** because the model is learning one language/text stream.

There is no English tokenizer + Tamil tokenizer here.

The tokenizer converts:

`Artificial intelligence is useful`

into:

`[token_id, token_id, token_id, token_id]`


In [18]:
# ============================================================
# 3. TOKENIZER
# ============================================================

MAX_VOCAB_SIZE = 5000

tokenizer = Tokenizer(
    # num_words=MAX_VOCAB_SIZE,
    filters="",
    oov_token="<unk>",
)

tokenizer.fit_on_texts(texts)
# print(vars(tokenizer))
# print((tokenizer.__dict__))
sequences = tokenizer.texts_to_sequences(texts)

vocab_size = min(
    len(tokenizer.word_index) + 1,
    MAX_VOCAB_SIZE,
)

print("Actual vocabulary:", len(tokenizer.word_index) + 1)
print("Vocabulary used by model:", vocab_size)


Actual vocabulary: 231
Vocabulary used by model: 231


## 4. Add `<eos>` and Create Token Sequences

In [21]:
# ============================================================
# 4. ADD EOS AND CREATE SEQUENCES
# ============================================================

EOS_TOKEN = "<eos>"

# Add <eos> so the model can learn when a generated sentence should end.
texts_with_eos = [
    text + " " + EOS_TOKEN
    for text in texts
]

tokenizer.fit_on_texts([EOS_TOKEN])

sequences = tokenizer.texts_to_sequences(texts_with_eos)

# Keep only tokens inside the model vocabulary.
sequences = [
    [token if token < vocab_size else tokenizer.word_index["<unk>"]
     for token in seq]
    for seq in sequences
]

print("Example text:")
print(texts_with_eos[0])
print()
print("Token IDs:")
print(sequences[0])


Example text:
Natural language processing helps build intelligent applications for document processing. Embeddings represent tokens as numerical vectors in a continuous space. <eos>

Token IDs:
[42, 11, 84, 21, 168, 52, 169, 5, 133, 134, 40, 98, 13, 99, 100, 101, 2, 3, 102, 103, 1]


## 5. Create Input → Next Token Target

In [24]:
# ============================================================
# 5. INPUT / TARGET CREATION
# ============================================================

# Example:
#
# Tokens:
# [Artificial, intelligence, is, useful, <eos>]
#
# Input:
# [Artificial, intelligence, is, useful]
#
# Target:
# [intelligence, is, useful, <eos>]
#
# The model learns to predict the NEXT token.

CONTEXT_LENGTH = 32

input_sequences = []
target_sequences = []

for seq in sequences:

    # Ignore sequences that are too short.
    if len(seq) < 2:
        continue

    # Create multiple training windows from longer text.
    for start in range(0, len(seq) - 1, CONTEXT_LENGTH):

        chunk = seq[start:start + CONTEXT_LENGTH + 1]

        if len(chunk) < 2:
            continue

        input_part = chunk[:-1]
        target_part = chunk[1:]

        input_sequences.append(input_part)
        target_sequences.append(target_part)

# Pad every training sequence to the same context length.
input_sequences = pad_sequences(
    input_sequences,
    maxlen=CONTEXT_LENGTH,
    padding="post",
    truncating="pre",
)

target_sequences = pad_sequences(
    target_sequences,
    maxlen=CONTEXT_LENGTH,
    padding="post",
    truncating="pre",
)

# Ignore padded target positions when calculating loss/accuracy.
sample_weight = (
    target_sequences != 0
).astype("float32")

print("Input shape:", input_sequences.shape)
print("Target shape:", target_sequences.shape)


Input shape: (500, 32)
Target shape: (500, 32)


## 6. Positional Embedding

In [25]:
# ============================================================
# 6. POSITIONAL EMBEDDING
# ============================================================

class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(
        self,
        vocab_size,
        sequence_length,
        d_model,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.token_embedding = Embedding(
            vocab_size,
            d_model,
            mask_zero=True,
        )

        self.position_embedding = Embedding(
            sequence_length,
            d_model,
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1,
        )

        positions = self.position_embedding(positions)
        tokens = self.token_embedding(inputs)

        return tokens + positions


## 7. Decoder-Only Transformer Block

Unlike the previous translation model, there is **no encoder and no cross-attention**.

The block only uses:

1. Causal self-attention
2. Residual connection + LayerNorm
3. Feed-forward network
4. Residual connection + LayerNorm

Causal attention means a token can look at the **past**, but not the future.


In [7]:
# ============================================================
# 7. DECODER-ONLY TRANSFORMER BLOCK
# ============================================================

class TransformerDecoderBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim,
        dropout=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model),
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout)
        self.dropout2 = Dropout(dropout)

    def call(
        self,
        x,
        training=False,
        attention_mask=None,
    ):

        # Causal self-attention.
        attn_output = self.self_attention(
            query=x,
            key=x,
            value=x,
            attention_mask=attention_mask,
            training=training,
        )

        x = self.norm1(
            x + self.dropout1(
                attn_output,
                training=training,
            )
        )

        # Feed-forward network.
        ffn_output = self.ffn(
            x,
            training=training,
        )

        x = self.norm2(
            x + self.dropout2(
                ffn_output,
                training=training,
            )
        )

        return x


## 8. Causal Mask

In [8]:
# ============================================================
# 8. CAUSAL MASK
# ============================================================

def causal_mask(length):

    # 1 = allowed to attend
    # 0 = blocked

    i = tf.range(length)[:, None]
    j = tf.range(length)[None, :]

    return tf.cast(
        i >= j,
        tf.int32,
    )

# Example for 4 tokens:
#
# [[1, 0, 0, 0],
#  [1, 1, 0, 0],
#  [1, 1, 1, 0],
#  [1, 1, 1, 1]]
#
# Token 1 cannot see tokens 2, 3, 4.
# Token 2 cannot see tokens 3, 4.


## 9. Decoder-Only Transformer

In [9]:
# ============================================================
# 9. DECODER-ONLY TRANSFORMER
# ============================================================

class Transformer(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_sequence_length,
        d_model=64,
        num_heads=4,
        ff_dim=128,
        dropout=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.embedding = PositionalEmbedding(
            vocab_size,
            max_sequence_length,
            d_model,
        )

        self.decoder = TransformerDecoderBlock(
            d_model,
            num_heads,
            ff_dim,
            dropout,
        )

        # One probability/logit for every vocabulary token.
        self.output_layer = Dense(vocab_size)

    def call(self, inputs, training=False):

        token_ids = inputs

        x = self.embedding(token_ids)

        # Padding mask.
        padding_mask = tf.cast(
            token_ids != 0,
            tf.int32,
        )[:, None, :]

        # Causal mask.
        sequence_length = tf.shape(token_ids)[1]

        cmask = causal_mask(
            sequence_length
        )[None, :, :]

        # Combine padding + causal masks.
        attention_mask = (
            cmask *
            tf.cast(
                padding_mask[:, None, :],
                tf.int32,
            )
        )

        x = self.decoder(
            x,
            training=training,
            attention_mask=attention_mask,
        )

        return self.output_layer(x)


## 10. Build the Small Text Generation Transformer

In [10]:
# ============================================================
# 10. BUILD MODEL
# ============================================================

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

transformer = Transformer(
    vocab_size=vocab_size,
    max_sequence_length=CONTEXT_LENGTH,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
)

# Build the model once.
dummy_input = tf.constant(
    input_sequences[:1],
    dtype=tf.int32,
)

dummy_logits = transformer(
    dummy_input,
    training=False,
)

transformer.summary()

print("Logits shape:", dummy_logits.shape)


Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ positional_embedding            │ ?                      │        16,832 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_block       │ ?                      │        33,472 │
│ (TransformerDecoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (1, 32, 231)           │        15,015 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,319 (255.15 KB)

 Trainable params: 65,319 (255.15 KB)

 Non-trainable params: 0 (0.00 B)

Logits shape: (1, 32, 231)


## 11. Compile

The target is an integer token ID.

Therefore we use:

`SparseCategoricalCrossentropy(from_logits=True)`

At every position the model predicts one probability/logit for every vocabulary token.


In [11]:
# ============================================================
# 11. COMPILE
# ============================================================

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

transformer.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=loss_fn,
    weighted_metrics=["accuracy"],
)


## 12. Train

In [12]:
# ============================================================
# 12. TRAIN
# ============================================================

EPOCHS = 100
BATCH_SIZE = 16

history = transformer.fit(
    input_sequences,
    target_sequences,
    sample_weight=sample_weight,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    verbose=1,
)


Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2736 - loss: 3.0295
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6209 - loss: 1.6147
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6623 - loss: 1.0189
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7173 - loss: 0.7772
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7598 - loss: 0.6416
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7831 - loss: 0.5446
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8040 - loss: 0.4792
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8095 - loss: 0.4361
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8120 - loss: 0.4097
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8139 - loss: 0.3928
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8149 - loss: 0.3801
Epoch 12/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy:

## 13. Autoregressive Text Generation + Sampling

During inference, the model predicts **one next token at a time**.

Example:

```text
Prompt: Machine learning

Machine learning
        ↓
predict next token
        ↓
Machine learning is
        ↓
predict next token
        ↓
Machine learning is widely
        ↓
...
```

The previous notebook used **greedy decoding**:

```python
next_id = tf.argmax(next_logits)
```

That always chooses the token with the highest probability.

Now we add three important Generative AI techniques:

### 🔥 1. Temperature

Controls how random the probability distribution is.

- Low temperature → safer / more predictable
- `temperature = 1.0` → normal distribution
- High temperature → more diverse / random

### 🔥 2. Top-K Sampling

Only the **K highest-probability tokens** are allowed to be selected.

Example:

```text
Top-K = 5

Vocabulary
10000 tokens
      ↓
keep only best 5
      ↓
sample one token
```

### 🔥 3. Top-P / Nucleus Sampling

Instead of keeping a fixed number of tokens, keep the **smallest group of tokens whose cumulative probability reaches P**.

Example:

```text
Top-P = 0.90

Token A = 0.50
Token B = 0.25
Token C = 0.10
Token D = 0.08
Token E = 0.07

A + B + C + D = 0.93

→ keep A, B, C, D
```

This adapts the number of candidate tokens according to the model's confidence.


In [13]:
# ============================================================
# 13. TEXT GENERATION + TEMPERATURE / TOP-K / TOP-P
# ============================================================

eos_id = tokenizer.word_index[EOS_TOKEN]

index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
    if index < vocab_size
}


# ------------------------------------------------------------
# 13.1 SAMPLE NEXT TOKEN
# ------------------------------------------------------------

def sample_next_token(
    logits,
    temperature=1.0,
    top_k=None,
    top_p=None,
):
    # logits shape:
    # [vocab_size]
    #
    # Higher logit = model thinks that token is more likely.

    logits = tf.cast(logits, tf.float32)

    # --------------------------------------------------------
    # TEMPERATURE
    # --------------------------------------------------------
    #
    # logits / temperature
    #
    # temperature < 1:
    #   makes high-probability tokens more dominant.
    #
    # temperature > 1:
    #   makes probabilities more spread out.
    #
    # temperature = 0:
    #   use greedy decoding.

    if temperature <= 0:
        return int(tf.argmax(logits).numpy())

    logits = logits / temperature

    # --------------------------------------------------------
    # TOP-K
    # --------------------------------------------------------
    #
    # Keep only the K highest logits.
    # All other tokens are removed by setting their logits
    # to -infinity.

    if top_k is not None and top_k > 0:
        k = min(top_k, int(logits.shape[-1]))

        values, _ = tf.math.top_k(
            logits,
            k=k
        )

        kth_value = values[-1]

        logits = tf.where(
            logits >= kth_value,
            logits,
            tf.fill(
                tf.shape(logits),
                tf.constant(-float("inf"), dtype=tf.float32)
            )
        )

    # --------------------------------------------------------
    # TOP-P / NUCLEUS SAMPLING
    # --------------------------------------------------------
    #
    # Sort tokens from highest probability to lowest.
    # Keep the smallest set whose cumulative probability
    # reaches top_p.

    if top_p is not None and 0 < top_p < 1.0:

        sorted_indices = tf.argsort(
            logits,
            direction="DESCENDING"
        )

        sorted_logits = tf.gather(
            logits,
            sorted_indices
        )

        sorted_probs = tf.nn.softmax(
            sorted_logits
        )

        cumulative_probs = tf.cumsum(
            sorted_probs
        )

        # Remove tokens after the probability mass crosses
        # top_p, while keeping the first token that crosses it.
        sorted_remove = cumulative_probs > top_p

        sorted_remove = tf.concat(
            [
                tf.constant([False]),
                sorted_remove[:-1],
            ],
            axis=0,
        )

        filtered_sorted_logits = tf.where(
            sorted_remove,
            tf.fill(
                tf.shape(sorted_logits),
                tf.constant(-float("inf"), dtype=tf.float32)
            ),
            sorted_logits,
        )

        # Put the filtered logits back in vocabulary order.
        logits = tf.scatter_nd(
            indices=tf.expand_dims(
                sorted_indices,
                axis=1
            ),
            updates=filtered_sorted_logits,
            shape=tf.shape(logits),
        )

    # --------------------------------------------------------
    # RANDOM SAMPLING
    # --------------------------------------------------------
    #
    # Convert logits to probabilities and randomly select
    # one token according to those probabilities.

    next_id = tf.random.categorical(
        tf.expand_dims(logits, axis=0),
        num_samples=1,
    )[0, 0]

    return int(next_id.numpy())


# ------------------------------------------------------------
# 13.2 GENERATE TEXT
# ------------------------------------------------------------

def generate_text(
    prompt,
    max_new_tokens=30,
    temperature=1.0,
    top_k=None,
    top_p=None,
):

    # Convert prompt into token IDs.
    prompt_ids = tokenizer.texts_to_sequences(
        [prompt]
    )[0]

    if not prompt_ids:
        return prompt

    # Replace IDs outside the model vocabulary with <unk>.
    unk_id = tokenizer.word_index["<unk>"]

    prompt_ids = [
        token if token < vocab_size else unk_id
        for token in prompt_ids
    ]

    generated_ids = prompt_ids.copy()

    for _ in range(max_new_tokens):

        # Use only the latest context window.
        context_ids = generated_ids[-CONTEXT_LENGTH:]

        model_input = pad_sequences(
            [context_ids],
            maxlen=CONTEXT_LENGTH,
            padding="post",
            truncating="pre",
        )

        logits = transformer(
            tf.constant(
                model_input,
                dtype=tf.int32,
            ),
            training=False,
        )

        # Number of actual tokens in this context.
        current_length = len(context_ids)

        # Predict the token after the last real token.
        next_logits = logits[
            0,
            current_length - 1,
        ]

        # Choose the next token using:
        # Temperature → Top-K → Top-P → Random Sampling
        next_id = sample_next_token(
            next_logits,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        # Stop when EOS is generated.
        if next_id == eos_id or next_id == 0:
            break

        generated_ids.append(next_id)

    words = []

    for token_id in generated_ids:

        if token_id == eos_id or token_id == 0:
            break

        words.append(
            index_to_word.get(
                token_id,
                "<unk>"
            )
        )

    return " ".join(words)


## 14. Compare Temperature, Top-K and Top-P

We will use the **same prompt** and change only the sampling settings.

This helps us understand how generation changes.

```text
temperature = 0
    ↓
Greedy decoding
    ↓
Most predictable output
```

```text
temperature = 0.7
top_k = 10
top_p = 0.9
    ↓
More controlled variation
```

```text
temperature = 1.2
top_k = 20
top_p = 0.95
    ↓
More diversity / randomness
```

Because our dataset is small, the generated text may still be repetitive or grammatically imperfect.


In [14]:
# ============================================================
# 14. TEST GENERATION WITH DIFFERENT SAMPLING SETTINGS
# ============================================================

prompt = "Artificial intelligence"

settings = [
    {
        "name": "Greedy",
        "temperature": 0,
        "top_k": None,
        "top_p": None,
    },
    {
        "name": "Temperature 0.7 + Top-K 10 + Top-P 0.9",
        "temperature": 0.7,
        "top_k": 10,
        "top_p": 0.9,
    },
    {
        "name": "Temperature 1.2 + Top-K 20 + Top-P 0.95",
        "temperature": 1.2,
        "top_k": 20,
        "top_p": 0.95,
    },
]

for setting in settings:

    print("=" * 70)
    print("Setting:", setting["name"])
    print("Prompt:", prompt)

    generated = generate_text(
        prompt,
        max_new_tokens=30,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        top_p=setting["top_p"],
    )

    print("Generated:", generated)


Setting: Greedy
Prompt: Artificial intelligence
Generated: artificial intelligence is widely used for improve decision making for enterprise applications. a tokenizer converts text into tokens that a neural network can process. <unk> neural network can process. <unk> predicts one
Setting: Temperature 0.7 + Top-K 10 + Top-P 0.9
Prompt: Artificial intelligence
Generated: artificial intelligence allows assist developers with programming for document processing. a tokenizer converts text into tokens that a neural network can process. <unk> neural network can process. <unk> neural network can help
Setting: Temperature 1.2 + Top-K 20 + Top-P 0.95
Prompt: Artificial intelligence
Generated: artificial intelligence enables translate text between languages in natural language applications. during generation, the model predicts one token at a time. <unk> time. <unk> time. <unk> time. <unk> time. <unk> time. <unk>


## 15. Test a New Prompt

Now try your own prompt.

You can change:

- `temperature`
- `top_k`
- `top_p`
- `max_new_tokens`

Try the same prompt several times with sampling enabled. Because sampling is random, the output can change between runs.


In [15]:
# ============================================================
# 15. ASK YOUR OWN PROMPT
# ============================================================

prompt = "Artificial intelligence"

print("Prompt:", prompt)

generated = generate_text(
    prompt,
    max_new_tokens=30,

    # Sampling controls
    temperature=0.7,
    top_k=10,
    top_p=0.9,
)

print("Generated:", generated)


Prompt: Artificial intelligence
Generated: artificial intelligence is widely used for improve decision making for enterprise applications. a tokenizer converts text into tokens that a neural network can process. <unk> neural network can process. <unk> process information


## 16. Learning Checkpoints

After this section, you should understand:

1. **Tokenizer** — converts text into token IDs.
2. **Next-token target** — teaches the model to predict the next token.
3. **Decoder-only Transformer** — uses causal self-attention.
4. **Causal mask** — prevents looking at future tokens.
5. **Autoregressive generation** — generates one token at a time.
6. **Greedy decoding** — always selects the highest-probability token.
7. **Temperature** — controls how sharp or flat the token probability distribution is.
8. **Top-K sampling** — limits sampling to the K most likely tokens.
9. **Top-P / Nucleus sampling** — limits sampling to the smallest probability set whose cumulative probability reaches P.
10. **Random sampling** — selects a token from the remaining probability distribution.

### Key idea

```text
Prompt
  ↓
Transformer
  ↓
Next-token probabilities
  ↓
Temperature
  ↓
Top-K
  ↓
Top-P
  ↓
Sample next token
  ↓
Append token to prompt
  ↓
Repeat
```

The model parameters are **not changing during generation**. Only the method used to choose the next token changes.
